

#### **Machine Learning and Data Science Research Project - Research**

**Project:** Reinforcement Learning for Optimal Execution 

**Author:** Robert de Witt (CID: 02541719)

In [ ]:
import pandas as pd
from notebook_logging_setup import quick_setup
from gymnasium.utils.env_checker import check_env
from stable_baselines3 import PPO
from gymnasium.wrappers import TimeLimit
import torch
import numpy as np
import random

# my libraries
from mkt_data_yfinance import MarketDataLoader as mdl
import render_plots as rp
import model_parking as mp
from optimal_execution_env import MultiOrderExecutionEnv as moee
from order_generator import OrderGenerator
import baseline_strategies as bs
import gpu_utils

quick_setup(debug=False)
fixed_seed = 42
np.random.seed(fixed_seed)
random.seed(fixed_seed)  # Also seed the random module

device = gpu_utils.get_device()
gpu_utils.set_random_seed(fixed_seed, device)

In [ ]:
max_trading_rate = 0.8
max_order_size_in_adv_pct = 0.1
tickers = ['AAPL','MSFT','GOOGL','AMZN','TSLA','NVDA','JPM','V','JNJ','DIS', 'NKE', 'PYPL', 'ADBE', 'NFLX', 'INTC', 'CMCSA', 'PEP', 'MRK', 'VZ', 'T']
md = mdl()
stock_df_list = md.load_data(tickers, horizon='7d')
stock_df_list = md.pre_process_data(stock_df_list)

# Check available dates
sample_df = stock_df_list[list(stock_df_list.keys())[0]]
dates = pd.to_datetime(sample_df.index.date).unique()
print(f"Available dates: {len(dates)} days from {dates[0]} to {dates[-1]}")

In [ ]:
# print first 3 rows of each stock dataframe
#for stock in stock_df_list:
#    print(stock_df_list[stock].head(-3))
# Create the environment

**Order Generation**

The order generation process creates a set of trading orders with the following characteristics:
1.  For each order, a random stock is selected from the available universe, and its Average Daily Volume (ADV) is calculated as the mean of daily trading volumes.
2. The order size is determined by sampling a random percentage of ADV between min_adv_pct and max_adv_pct, scaled by the time horizon (as a fraction of the trading day). This ensures that larger orders are spread over longer time periods and that the orders can complete within the time horizon given 
3. The execution window is defined by randomly selecting a start time and a time horizon (in minutes) between min_time_horizon and max_time_horizon, with the end time being start_time + time_horizon.
4. Each order is assigned a random side (buy/sell) and its quantity is rounded to the nearest integer, with safeguards to prevent zero-sized orders. The resulting orders are then used to train or evaluate the execution strategy in the reinforcement learning environment.
5. For simplicity's sake, we will not apply order constraints such as limit prices or volume limits, we will assume lit market trading only and we will not worry about auctions.

In [ ]:
# Create train/test split using date filtering
# Training set: first 5 days (exclude last 2 days)
train_orders_df = OrderGenerator(
    stock_df_list=stock_df_list, 
    market_data=md, 
    debug=False, 
    max_adv_pct=max_order_size_in_adv_pct, 
    num_orders=10000,
    sd_delta=0,  # Start from first day
    ed_delta=2,   # Exclude last 2 days
    seed=fixed_seed
).get_orders()

# Test set: last 2 days (exclude first 5 days)
test_orders_df = OrderGenerator(
    stock_df_list=stock_df_list, 
    market_data=md, 
    debug=False, 
    max_adv_pct=max_order_size_in_adv_pct, 
    num_orders=2000,
    sd_delta=5,  # Skip first 5 days
    ed_delta=0,   # Include up to last day
    seed=fixed_seed
).get_orders()

# Verify the date split
print(f"\nTrain set: {len(train_orders_df)} orders")
print(f"Train dates: {train_orders_df['date'].min()} to {train_orders_df['date'].max()}")
print(f"Orders per day (train):\n{train_orders_df['date'].value_counts().sort_index()}")

print(f"\nTest set: {len(test_orders_df)} orders")
print(f"Test dates: {test_orders_df['date'].min()} to {test_orders_df['date'].max()}")
print(f"Orders per day (test):\n{test_orders_df['date'].value_counts().sort_index()}")

# Verify no overlap
train_dates = set(train_orders_df['date'].unique())
test_dates = set(test_orders_df['date'].unique())
print(f"\nDate overlap: {train_dates.intersection(test_dates)}")
print(f"No overlap verified: {len(train_dates.intersection(test_dates)) == 0}")

In [ ]:
# print orders dataframe in single line per order
rp.display_order_info(train_orders_df, name='Sample Train', num_orders=3)
rp.display_order_info(test_orders_df, name='Sample Test', num_orders=3)
rp.plot_order_histograms(train_orders_df, test_orders_df)

# Show sample orders with dates
print("\nSample train orders with dates:")
print(train_orders_df[['ticker', 'date', 'start_time', 'end_time', 'order_qty', 'side']].head(5))
print("\nSample test orders with dates:")
print(test_orders_df[['ticker', 'date', 'start_time', 'end_time', 'order_qty', 'side']].head(5))


**Order Characteristics**

Most of the orders are naturally stratified on size, stock, side and horizon. In the next version of the order generator, we will take a much wider selection of stocks to capture different levels of liquidity, volatility, trending markets, shocks, stable markets, etc.

**RL Environment**

In the current setup the agent’s **state** at each step is a 14-dimensional real vector  
$$
s_t = \bigl[p_t^{\rm mid},\,V_t,\,T - t,\,q_t,\,\alpha,\,x,\,f,\,m,\,\delta,\,A,\,p_0,\,r,\,v^{(1)},\,v^{(5)}\bigr],
$$  
where it sees the current mid-price, market volume, time and shares remaining, target participation rate, trading signals, last fill and trade stats, arrival price, regime and short/long‐term volatilities.  Its **action**  
$$
a_t \in \{-1.0,\,-0.75,\, -0.5,\,-0.25,\,-0.1,\,0,\,0.10,\,0.15,\,0.25,\,0.50,\,0.75,\ 1.0\}
$$  
is “accelerate or decelerate versus the constant rate to trade this fraction of today’s volume,” so the env executes  
$$
\Delta q_t = \min(\rho_i(1 + a_t)\,V_t,\;q_t)
$$  
shares (and updates  
$$
q_{t+1} = q_t - \Delta q_t
$$  

$$\rho_i = \frac{Q_i}{V[t_{start}, t_{end}]}$$ 
and the running VWAP).  Episodes end when either all shares are filled or the time horizon is hit, a large penalty is applied if the order does not complete.


**Trading Simulation**

In order to realistically simulate market behaviour, each trade needs to create an impact on price, and successive trades must propagate impact to the next order and so on. Rather than consider the nuance of trade by trade, limit and market orders and order books which require many assumptions and complex limit order book simulators, we focus on minutely percentage of volume trading. This allows to focus on building an optimal scheduling algorithm and removes the granular complexities of limit order books, order types and venue types.

For minutely impact level impact estimations have utilized the transient impact model 2 from Bouchaud's book Trades, Quotes and Prices [7] which is parameterized with intuitive default values for this update and assumes the 1 minute trade impact follows square root.  We account for gaps between 1 minute trade intervals allowing for more accurate decay. In the final report, we will include a calibrated version of this model based on the trades observed via Bank of America's trade data. 


**Linear Transient‐Impact (Propagator) Model [7]**  
$$
r_t \;=\;\sum_{t'<t} G(t - t')\,\epsilon_{t'} \;+\;\eta_t,
$$
where $r_t = P_t - P_{t-1}$ is the mid-price return, $\epsilon_{t'}\!=\pm1$ the sign of the trade at time $t'$, $G(\tau)$ the propagator kernel, and $\eta_t$ an uncorrelated noise term.  

In our case we do allow for non-uniformity of time steps between trades so we modify to consider the $t_{\Delta}$:

$$
\lambda(\Delta t)
\;=\;
\exp\!\Bigl(-\frac{\Delta t}{\tau}\Bigr),
\quad
\Delta I_t
\;=\;
Y\,\sigma\,\sqrt{\frac{q_t}{V}},
$$

$$
I_t
\;=\;
\lambda(\Delta t)\,I_{t-1}
\;+\;
\Delta I_t,
$$

$$
p_t^{\mathrm{fill}}
\;=\;
p_t^{\mathrm{vwap}}
\bigl(1 \;+\; s\,I_t\bigr).
$$

In [ ]:
# generate the execution environments for training
train_execution_env = moee(
    stock_df_list=stock_df_list, 
    orders_df=train_orders_df, 
    impact_coef=1, 
    num_envs=1, 
    decay_rate=5, 
    window_size=1, 
    min_rate=0, 
    max_rate=max_trading_rate,
    seed=fixed_seed  # Use fixed seed for reproducibility
)
max_horizon = int(train_orders_df["time_horizon"].max())
train_env = TimeLimit(train_execution_env, max_episode_steps=max_horizon)
check_env(train_env)
print("✓ Train environment determinism check passed!")

# generate the execution environments for testing
test_execution_env = moee(
    stock_df_list=stock_df_list, 
    orders_df=test_orders_df, 
    impact_coef=1, 
    num_envs=1, 
    decay_rate=5, 
    window_size=1, 
    min_rate=0, 
    max_rate=max_trading_rate,
    seed=fixed_seed
)
max_horizon = int(test_orders_df["time_horizon"].max())
test_env = TimeLimit(test_execution_env, max_episode_steps=max_horizon)
check_env(test_env)

**Agents**

This is a basic "out of the box" PPO agent with no tuning or modification. The purpose is the allow for a constant at the environment is being bult.

In [ ]:
# set up the environment for training
mp = mp.ModelParking()

In [ ]:
# PPO models
model_name = "ppo_multiorder_exec"
print (f"Training model: {model_name}")
model = PPO("MlpPolicy", train_env, device="cpu", verbose=0, learning_rate=3e-4, 
            n_steps=512, batch_size=256, n_epochs=10,gamma=0.999, gae_lambda=0.95, clip_range=0.5,
            policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256]), activation_fn=torch.nn.ReLU))
print(f"Parking model, learning and saving: {model_name}")
mp.park_model(model, model_name, learn=True, steps=4000)


model_name = "ppo_multiorder_double_exec"
print (f"Training model: {model_name}")
model_double = PPO("MlpPolicy", train_env, device="cpu", verbose=0, learning_rate=3e-4, 
            n_steps=512, batch_size=512, n_epochs=50,gamma=0.999, gae_lambda=0.95, clip_range=0.5,
            policy_kwargs=dict(net_arch=dict(pi=[512, 512], vf=[512, 512]), activation_fn=torch.nn.ReLU))
print(f"Parking model, learning and saving: {model_name}")
mp.park_model(model_double, model_name, learn=True, steps=8000)



In [ ]:
# Baseline comparison strategies, VWAP and Random
model_name = "vwap_baseline"
print (f"Loading model: {model_name}")
vwap_model = bs.VWAPBaseline(train_env)
print(f"Parking model, learning and saving: {model_name}")
mp.park_model(vwap_model, model_name, learn=False, save=False)

model_name = "random_baseline"
print (f"Loading model: {model_name}")
random_model = bs.RandomBaseline(train_env)
print(f"Parking model, learning and saving: {model_name}")
mp.park_model(random_model, model_name, learn=False, save=False)



In [ ]:
model_name_list = mp.list_models()
print(f"Available models: {model_name_list}")

train_orders = {}
test_orders = {}
num_episodes = 20

for model_name in model_name_list:
    print(f"Executing orders with model: {model_name}")
    model = mp.get_model(model_name)
    
    # Check if the model is loaded correctly
    if model is None:
        print(f"Model {model_name} could not be loaded.")
        continue
    
    # Execute orders in the training environment
    if isinstance(model, (bs.VWAPBaseline, bs.RandomBaseline)):
        print(f"Skipping training execution for model {model_name} as it is not a valid model type.")
    else:
        print(f"Executing {num_episodes} orders in the training environment...")
        train_orders[model_name] = train_execution_env.execute_orders(model=model, num_episodes=num_episodes)

    # Execute orders in the test environment
    print(f"Executing {num_episodes} orders in the test environment...")
    test_orders[model_name] = test_execution_env.execute_orders(model=model, num_episodes=num_episodes)


In [ ]:
orders_by_minute_df = pd.DataFrame(columns=['model','episode', 'order_idx', 'ticker', 'side', 'order_qty', 'adv_pct', 'start_time', 'end_time', 'time_horizon'])
print(f"Executed {len(test_orders)} models:")
for model in test_orders:
    print(f"Model: {model}, Episodes: {len(test_orders[model])}")
    for episode_idx, episode in enumerate(test_orders[model]):
        print(f"  Episode {episode_idx+1}: {len(episode)} steps")
        for order_idx, order in enumerate(episode):
            order['model'] = model
            order['episode'] = episode_idx
            order['order_idx'] = order_idx
            orders_by_minute_df = pd.concat([orders_by_minute_df, pd.DataFrame([order])], ignore_index=True)

In [ ]:
# To display a single order's time steps
pd.set_option('display.max_columns', None)
orders_by_minute_df[(orders_by_minute_df['episode'] == 3) & (orders_by_minute_df['model'] == 'ppo_multiorder_double_exec')][0:]

**Evaluation**


In [ ]:
rp.plot_orders(test_orders, num_orders=3)
rp.create_execution_summary_table(orders_dict=train_orders,  trim=0.01)
rp.create_execution_summary_table(orders_dict=test_orders,  trim=0.01)

Note: there is an error in the dislayed reward calculation.

**Execution Summmary Table Observations**

At this stage, we have built the foundations of what we hope will be a robust framework to extend for trialling agents. it is clear that we need to focus on tuning the agents and the environment. Despite having high impact on price, the agent continually chooses much higher percentages of volume causing the orders to complete within 20% of the order horizons on average for no perceived price advantage. We will experiment with penalizing the agent for deviating too highly from an expected percentage of volume and/or narrowing the action space to close to the target POV. We wll also add VWAP and an optimal arrival benchmark to encourage more temporal exploration. We also look forward to looking throgh the to-dos below and making them happen in the final project.

**References**

[1] Lehman, J. & Stanley, K. O. (2011). _Evolving a Diversity of Creatures through Novelty Search and Local Competition_. In Proceedings of the 13th Genetic and Evolutionary Computation Conference (GECCO) (pp. 211–218). ACM.

[2] Pugh, J. K., Soros, L. B., & Stanley, K. O. (2016). _Quality Diversity: A New Frontier for Evolutionary Computation_. Frontiers in Robotics and AI, 3, 40.

[3] Cully, A. & Demiris, Y. (2017). _Quality and Diversity Optimization: A Unifying Modular Framework_. arXiv:1708.09251.

[4] Mouret, J.-B. & Clune, J. (2015). _Illuminating Search Spaces by Mapping Elites_. arXiv:1504.04909.

[5] Cully, A., Clune, J., Tarapore, D., & Mouret, J.-B. (2015). _Robots That Can Adapt Like Animals_. Nature, 521(7553), 503–507.

[6] Gaier, A., Asteroth, A., & Mouret, J.-B. (2017). _Data-Efficient Exploration, Optimization, and Modeling of Diverse Designs through Surrogate-Assisted Illumination_. arXiv:1702.03713.

[7] Bouchaud, J.-P., Bonart, J., Donier, J., & Gould, M. (2018). _The Propagator Model_. In Trades, Quotes and Prices: Financial Markets Under the Microscope (pp. 252–253). Cambridge University Press.

[8] Tóth, B., Lempérière, Y., Deremble, C., de Lataillade, J., Kockelkoren, J., & Bouchaud, J.-P. (2011). _Anomalous price impact and the critical nature of liquidity in financial markets_. Physical Review X, 1(2), 021006.









**Appendix**

To-dos for the final project:

* Can we speed up the execution code? - use VecEnv instead of gym.Env
    * Create parallelizable agent testing framework?
* Ensure all comparisons use the same set of orders and they are plotted

* What about adding tensorboard?
* Fix VWAP baseline to use volume profile and execute the full horizon

* Load in mana market data 
* Add longer look-back on market data
* Calibrate impact model by stock with mana data for tau, Y
-------------------------------------------------------------------
* Find and break out days by different regimes, e.g. high volatility, low volatility, trending, mean reverting, shocks, stable markets, etc. add these to the gym
* Consider more advanced propagator models such as the 6-event propagator model (ch. 14 of Trades, Quotes and Prices)
* Implement multiple agents from papers and using Quality Diveristy and MAP methods
* Linear versus square root for 1 minute impact?
* Make volume profiles to scale volume and remove auctions, remove special days
* TODO: fix the warnings in check env
* TODO: fix warnings in generate orders
* TODO: refactor optimal execuion env
